# Cross-acquisition: four remaining runs

B-training/A-testing seeds 42,123,2026 plus A-training/B-testing seed2026. Uses the recheck KMeans/sklearn protocol. Two earlier A-training/B-testing runs are separate records.

Source: `gd_cross_acquisition_remaining_runs.py`.

Run in a fresh Kaggle GPU session with the appropriate dataset attached. Review the configuration near the top; original dataset/output paths are retained. Do not Run All across the repository.


In [ ]:
"""
GD cross-acquisition: FOUR new experiments to complete SIX direction/seed results.

Default new runs:
    B (183) -> A (217): seeds 42, 123, 2026
    A (217) -> B (183): seed 2026 only
Existing A->B seeds 42/123 are NOT retrained and their numbers are NOT hard-coded.

Derived from gd-second.ipynb's saved A->B recheck cell (zero-based cell 12).
Preserves its model, loss, augmentation, preprocessing, sklearn development
split, seeded loaders, 70-epoch cap, validation selection and fresh refit.
The earlier bidirectional cell uses a different development split implementation;
this file deliberately follows the RECHECK implementation for all new runs.

Each experiment has TWO phases: source-only development training, then a fresh
refit on all source images for the validation-selected epoch count. Target
images are used only for final no-TTA evaluation. The internal development
split is random within the source group; it is not a buffered spatial split.

Use Kaggle with GPU and attach GD-RGBMS-DB. Paste this file into one code cell,
or execute %run /path/to/gd_cross_acquisition_remaining_runs.py.
Dependencies: numpy, Pillow, tqdm, torch (with torch.amp), scikit-learn.
Colab: set DATASET_ROOT and mount Drive yourself if using Drive-backed paths.

Outputs: exact group/split lists, full-precision global and per-image metrics,
TP/FP/FN/TN, development and final checkpoints, epoch logs, configuration,
per-run ZIPs, and a combined CSV updated after each completed experiment.
Completed runs in the SAME output directory are skipped after signature checks.
An interrupted unfinished run restarts from epoch 1 (no epoch-level resume).
Download results before ending a Kaggle session; local storage is not durable.
"""
from __future__ import annotations

import csv
import json
import math
import random
import re
import shutil
import time
from pathlib import Path

import numpy as np
from PIL import Image, ExifTags
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split



import contextlib
import gc
import hashlib
import sys
import traceback

# ------------------------ USER SETTINGS ------------------------
DATASET_ROOT = Path("/kaggle/input/datasets/hemal7/gd-rgbms-db")
WORK_DIR = (Path("/kaggle/working") if Path("/kaggle/working").exists()
            else Path("/content") if Path("/content").exists() else Path.cwd())
MASTER_OUT = WORK_DIR / "gd_cross_acquisition_remaining_runs"
RUN_PLAN = [
    ("B_to_A", 42),
    ("B_to_A", 123),
    ("B_to_A", 2026),
    ("A_to_B", 2026),
]
SKIP_COMPLETED = True
REQUIRE_CUDA = True
# Optional: official final metrics CSVs from the two existing A->B runs.
# No remembered, inferred or reference-only numbers are imported automatically.
EXISTING_RESULT_CSVS = []  # e.g. [Path("/kaggle/input/old-run/final_test_metrics.csv")]
# ---------------------------------------------------------------

TARGET_SIZE = (896, 672)  # width, height
BATCH_SIZE = 2
ACCUMULATION_STEPS = 2
MAX_EPOCHS = 70
EARLY_STOPPING_PATIENCE = 12

LEARNING_RATE = 1e-4
WEIGHT_DECAY = 1e-4
LR_REDUCTION_FACTOR = 0.5
LR_SCHEDULER_PATIENCE = 4
MIN_LEARNING_RATE = 1e-6

NUM_WORKERS = 0
SHOW_PROGRESS = False
CACHE_SOURCE_GROUP = True

BRANCH_BASE = 28
PREFERRED_GN_GROUPS = 8
MODALITY_DROPOUT_PROBABILITY = 0.0

RGB_CHANNELS = 3
SPECTRAL_CHANNELS = 3
TOTAL_INPUT_CHANNELS = 6
INPUT_CHANNEL_ORDER = ("R", "G", "B", "Green", "Red-Edge", "NIR")

MASK_POSITIVE_RGB = np.array([240, 128, 128], dtype=np.int16)

AUX_RGB_INITIAL_WEIGHT = 0.15
AUX_SPECTRAL_INITIAL_WEIGHT = 0.25
AUX_RGB_MIDDLE_WEIGHT = 0.10
AUX_SPECTRAL_MIDDLE_WEIGHT = 0.15
AUX_RGB_FINAL_WEIGHT = 0.05
AUX_SPECTRAL_FINAL_WEIGHT = 0.08
AUX_MIDDLE_PHASE_START = 0.40
AUX_FINAL_PHASE_START = 0.70

COARSE_THRESHOLDS = np.round(np.arange(0.20, 0.851, 0.05), 2).tolist()
FINE_THRESHOLDS = np.round(np.arange(0.20, 0.801, 0.01), 2).tolist()


try:
    BILINEAR = Image.Resampling.BILINEAR
    NEAREST = Image.Resampling.NEAREST
except AttributeError:
    BILINEAR = Image.BILINEAR
    NEAREST = Image.NEAREST


# =============================================================================
# 2. REPRODUCIBILITY / DEVICE
# =============================================================================

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True



def run_experiment(direction, seed, out_dir, run_signature):
    """Original recheck execution, parameterized only by direction/seed/paths."""
    SEED = int(seed)
    OUT_DIR = out_dir
    OUT_DIR.mkdir(parents=True, exist_ok=True)
    seed_everything(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    pin_memory = device.type == "cuda"


    # =============================================================================
    # 3. DATASET DISCOVERY
    # =============================================================================

    def is_dataset_root(path: Path):
        return (
            (path / "RGB").is_dir()
            and (path / "MS").is_dir()
            and (path / "Mark").is_dir()
        )


    def find_dataset_root():
        if DATASET_ROOT.exists() and is_dataset_root(DATASET_ROOT):
            return DATASET_ROOT

        roots = [
            Path("/kaggle/input"),
            Path("/content"),
            Path("/content/drive/MyDrive"),
        ]

        candidates = []

        for root in roots:
            if not root.exists():
                continue

            for rgb_folder in root.rglob("RGB"):
                candidate = rgb_folder.parent
                if is_dataset_root(candidate):
                    candidates.append(candidate)

        if not candidates:
            raise FileNotFoundError(
                "Could not locate GD-RGBMS-DB. Expected folders RGB/, MS/, Mark/. "
                "Set DATASET_ROOT manually if needed."
            )

        candidates.sort(
            key=lambda p: (
                "gd-rgbms" not in str(p).lower(),
                len(str(p)),
            )
        )

        return candidates[0]


    BASE_DIR = find_dataset_root()
    RGB_DIR = BASE_DIR / "RGB"
    MS_DIR = BASE_DIR / "MS"
    MARK_DIR = BASE_DIR / "Mark"


    def sample_stem(name: str):
        value = Path(str(name)).name

        for extension in (
            ".JPG", ".jpg", ".JPEG", ".jpeg",
            ".PNG", ".png", ".TIF", ".tif",
        ):
            if value.lower().endswith(extension.lower()):
                return value[:-len(extension)]

        return value


    def find_path(directory: Path, stem: str, extensions):
        for ext in extensions:
            p = directory / f"{stem}{ext}"
            if p.exists():
                return p

        # Case-insensitive fallback.
        matches = [
            p for p in directory.glob(f"{stem}.*")
            if p.is_file()
        ]

        if matches:
            return matches[0]

        raise FileNotFoundError(f"Could not find paired file for {stem} in {directory}")


    rgb_files = sorted([
        p for p in RGB_DIR.iterdir()
        if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"}
    ])

    if len(rgb_files) != 400:
        print(
            f"WARNING: found {len(rgb_files)} RGB files; "
            "the previous run used 400 total samples."
        )

    ALL_NAMES = [sample_stem(p.name) for p in rgb_files]

    if len(ALL_NAMES) != len(set(ALL_NAMES)):
        raise RuntimeError("Duplicate RGB stems detected.")


    # =============================================================================
    # 4. GPS EXTRACTION
    # =============================================================================

    def _rational_float(value):
        try:
            return float(value)
        except Exception:
            pass

        numerator = getattr(value, "numerator", None)
        denominator = getattr(value, "denominator", None)

        if numerator is not None and denominator:
            return float(numerator) / float(denominator)

        if isinstance(value, tuple) and len(value) == 2:
            return float(value[0]) / float(value[1])

        raise TypeError(f"Cannot convert EXIF rational value: {value!r}")


    def _dms_to_decimal(dms, ref):
        degrees = _rational_float(dms[0])
        minutes = _rational_float(dms[1])
        seconds = _rational_float(dms[2])

        decimal = degrees + minutes / 60.0 + seconds / 3600.0

        if str(ref).upper() in {"S", "W"}:
            decimal = -decimal

        return decimal


    def gps_from_exif(path: Path):
        """
        Read latitude/longitude from RGB EXIF using Pillow.
        Returns (lat, lon) or None.
        """
        try:
            with Image.open(path) as image:
                exif = image.getexif()

                if not exif:
                    return None

                gps_data = None

                # Modern Pillow path.
                try:
                    gps_ifd_tag = getattr(ExifTags, "IFD", None)
                    if gps_ifd_tag is not None:
                        gps_data = exif.get_ifd(ExifTags.IFD.GPSInfo)
                except Exception:
                    gps_data = None

                # Compatibility path.
                if not gps_data:
                    gps_tag_id = None

                    for tag_id, tag_name in ExifTags.TAGS.items():
                        if tag_name == "GPSInfo":
                            gps_tag_id = tag_id
                            break

                    if gps_tag_id is not None:
                        raw_gps = exif.get(gps_tag_id)

                        if isinstance(raw_gps, dict):
                            gps_data = raw_gps
                        elif raw_gps is not None:
                            try:
                                gps_data = exif.get_ifd(gps_tag_id)
                            except Exception:
                                gps_data = None

                if not gps_data:
                    return None

                decoded = {}

                for key, value in gps_data.items():
                    decoded[ExifTags.GPSTAGS.get(key, key)] = value

                lat_dms = decoded.get("GPSLatitude")
                lat_ref = decoded.get("GPSLatitudeRef")
                lon_dms = decoded.get("GPSLongitude")
                lon_ref = decoded.get("GPSLongitudeRef")

                if None in (lat_dms, lat_ref, lon_dms, lon_ref):
                    return None

                latitude = _dms_to_decimal(lat_dms, lat_ref)
                longitude = _dms_to_decimal(lon_dms, lon_ref)

                return float(latitude), float(longitude)

        except Exception:
            return None


    def gps_from_filename(stem: str):
        """
        Fallback for filenames that already contain lat/lon.
        """
        patterns = [
            r"lat[_=-]?(-?\d+(?:\.\d+)?).*?(?:lon|lng|long)[_=-]?(-?\d+(?:\.\d+)?)",
            r"latitude[_=-]?(-?\d+(?:\.\d+)?).*?longitude[_=-]?(-?\d+(?:\.\d+)?)",
        ]

        for pattern in patterns:
            match = re.search(pattern, stem, flags=re.IGNORECASE)
            if match:
                return float(match.group(1)), float(match.group(2))

        return None


    gps_rows = []

    for rgb_path, stem in zip(
        tqdm(rgb_files, desc="Reading GPS", disable=not SHOW_PROGRESS),
        ALL_NAMES,
    ):
        gps = gps_from_exif(rgb_path)

        if gps is None:
            gps = gps_from_filename(stem)

        if gps is None:
            raise RuntimeError(
                f"No GPS found for {rgb_path.name}. "
                "The cross-acquisition split requires GPS coordinates."
            )

        gps_rows.append({
            "sample_name": stem,
            "latitude": gps[0],
            "longitude": gps[1],
            "rgb_path": str(rgb_path),
        })


    # =============================================================================
    # 5. TWO ACQUISITION GROUPS
    # =============================================================================

    latitudes = np.array([r["latitude"] for r in gps_rows], dtype=np.float64)
    longitudes = np.array([r["longitude"] for r in gps_rows], dtype=np.float64)

    EARTH_RADIUS_M = 6_371_000.0
    mean_latitude_rad = np.deg2rad(latitudes.mean())

    x_m = (
        np.deg2rad(longitudes - longitudes.mean())
        * EARTH_RADIUS_M
        * np.cos(mean_latitude_rad)
    )

    y_m = (
        np.deg2rad(latitudes - latitudes.mean())
        * EARTH_RADIUS_M
    )

    xy = np.column_stack([x_m, y_m])

    kmeans = KMeans(
        n_clusters=2,
        random_state=SEED,
        n_init=20,
    )

    raw_labels = kmeans.fit_predict(xy)

    cluster_0_names = [
        gps_rows[i]["sample_name"]
        for i in range(len(gps_rows))
        if raw_labels[i] == 0
    ]

    cluster_1_names = [
        gps_rows[i]["sample_name"]
        for i in range(len(gps_rows))
        if raw_labels[i] == 1
    ]

    # Reproduce the previous naming convention:
    # the 217-image acquisition was called A and the 183-image acquisition B.
    if len(cluster_0_names) >= len(cluster_1_names):
        GROUP_A = sorted(cluster_0_names)
        GROUP_B = sorted(cluster_1_names)
        a_raw_label, b_raw_label = 0, 1
    else:
        GROUP_A = sorted(cluster_1_names)
        GROUP_B = sorted(cluster_0_names)
        a_raw_label, b_raw_label = 1, 0

    a_points = xy[raw_labels == a_raw_label]
    b_points = xy[raw_labels == b_raw_label]

    pairwise_ab = np.sqrt(
        ((a_points[:, None, :] - b_points[None, :, :]) ** 2).sum(axis=2)
    )

    minimum_ab_distance_m = float(pairwise_ab.min())

    print("=" * 100)
    print(f"CROSS-ACQUISITION GENERALIZATION — {direction} | seed {SEED}")
    print("=" * 100)
    print("Dataset:", BASE_DIR)
    print("Device :", device)
    print("Seed   :", SEED)
    print("Input  :", TARGET_SIZE, "| batch", BATCH_SIZE, "| accumulation", ACCUMULATION_STEPS)
    print()
    print("GPS acquisition groups:")
    print("  Group A:", len(GROUP_A))
    print("  Group B:", len(GROUP_B))
    print(f"  Minimum A-B centre distance: {minimum_ab_distance_m:.3f} m")

    if len(GROUP_A) != 217 or len(GROUP_B) != 183:
        raise RuntimeError(f"Expected A=217, B=183; got A={len(GROUP_A)}, B={len(GROUP_B)}.")

    group_manifest = {"A": GROUP_A, "B": GROUP_B}
    group_file = MASTER_OUT / "locked_acquisition_groups.json"
    if group_file.exists():
        if json.loads(group_file.read_text()) != group_manifest:
            raise RuntimeError("Acquisition membership changed between seeds; inspect GPS groups before training.")
    else:
        group_file.write_text(json.dumps(group_manifest, indent=2))

    # Save GPS/group assignments.
    with open(
        OUT_DIR / "cross_acquisition_groups.csv",
        "w",
        newline="",
        encoding="utf-8",
    ) as f:
        writer = csv.DictWriter(
            f,
            fieldnames=["sample_name", "latitude", "longitude", "group"],
        )
        writer.writeheader()

        group_lookup = {
            **{name: "A" for name in GROUP_A},
            **{name: "B" for name in GROUP_B},
        }

        for row in gps_rows:
            writer.writerow({
                "sample_name": row["sample_name"],
                "latitude": row["latitude"],
                "longitude": row["longitude"],
                "group": group_lookup[row["sample_name"]],
            })


    # =============================================================================
    # 6. SOURCE-ONLY DEVELOPMENT SPLIT
    # =============================================================================

    source_group, target_group = direction.split("_to_")
    source_names = GROUP_A if source_group == "A" else GROUP_B
    target_names = GROUP_B if target_group == "B" else GROUP_A
    assert set(source_names).isdisjoint(target_names)
    assert len(source_names) + len(target_names) == 400

    development_validation_count = int(round(0.20 * len(source_names)))

    dev_train_names, dev_val_names = train_test_split(
        source_names,
        test_size=development_validation_count,
        random_state=SEED,
        shuffle=True,
    )

    dev_train_names = list(dev_train_names)
    dev_val_names = list(dev_val_names)

    print("\n" + "#" * 100)
    print(f"# {direction}: TRAIN/DEVELOP ON {source_group} -> TEST ON {target_group} | seed {SEED}")
    print("#" * 100)
    print("Source acquisition images     :", len(source_names))
    print("  Development train           :", len(dev_train_names))
    print("  Development validation      :", len(dev_val_names))
    print("Final refit train (ALL source):", len(source_names))
    print("Final untouched test          :", len(target_names))

    assert len(dev_train_names) == len(source_names) - development_validation_count
    assert len(dev_val_names) == development_validation_count
    assert set(dev_train_names).isdisjoint(dev_val_names)
    assert set(dev_train_names) | set(dev_val_names) == set(source_names)
    for filename, names in (
        ("development_train_list.txt", dev_train_names),
        ("development_val_list.txt", dev_val_names),
        ("final_train_all_source_list.txt", source_names),
        ("final_test_heldout_acquisition_list.txt", target_names),
    ):
        (OUT_DIR / filename).write_text("\n".join(names) + "\n")


    # =============================================================================
    # 7. RAW IMAGE / MASK LOADING
    # =============================================================================

    def binary_mask(image):
        array = np.asarray(image.convert("RGB"), dtype=np.int16)

        return (
            np.abs(array - MASK_POSITIVE_RGB).sum(axis=2) < 80
        ).astype(np.float32)


    def load_raw(name):
        stem = sample_stem(name)

        rgb_path = find_path(
            RGB_DIR,
            stem,
            [".JPG", ".jpg", ".JPEG", ".jpeg", ".PNG", ".png"],
        )

        ms_path = find_path(
            MS_DIR,
            stem,
            [".PNG", ".png", ".JPG", ".jpg"],
        )

        mask_path = find_path(
            MARK_DIR,
            stem,
            [".png", ".PNG", ".jpg", ".JPG"],
        )

        with Image.open(rgb_path) as item:
            rgb_image = item.convert("RGB").resize(TARGET_SIZE, BILINEAR)

        with Image.open(ms_path) as item:
            ms_image = item.convert("RGB").resize(TARGET_SIZE, BILINEAR)

        with Image.open(mask_path) as item:
            mask_image = item.convert("RGB").resize(TARGET_SIZE, NEAREST)

        rgb = np.asarray(rgb_image, dtype=np.float32).copy() / 255.0
        ms = np.asarray(ms_image, dtype=np.float32).copy() / 255.0
        mask = binary_mask(mask_image)

        image = np.concatenate([rgb, ms], axis=2).astype(np.float32)

        if image.shape != (TARGET_SIZE[1], TARGET_SIZE[0], 6):
            raise RuntimeError(
                f"Unexpected input shape {image.shape} for {stem}."
            )

        return image, mask, stem


    # =============================================================================
    # 8. CACHE ALL SOURCE IMAGES + COMPUTE DEV-TRAIN NORMALIZATION
    # =============================================================================

    SOURCE_CACHE = {}

    channel_sum = np.zeros(6, dtype=np.float64)
    channel_square_sum = np.zeros(6, dtype=np.float64)
    pixel_count = 0

    dev_train_set = set(dev_train_names)

    for name in tqdm(
        source_names,
        desc=f"{direction}: cache source + train stats",
        disable=not SHOW_PROGRESS,
    ):
        image, mask, stem = load_raw(name)

        if CACHE_SOURCE_GROUP:
            SOURCE_CACHE[name] = (
                image.astype(np.float16),
                mask.astype(np.uint8),
                stem,
            )

        if name in dev_train_set:
            channel_sum += image.sum(axis=(0, 1), dtype=np.float64)
            channel_square_sum += np.square(
                image,
                dtype=np.float32,
            ).sum(axis=(0, 1), dtype=np.float64)

            pixel_count += image.shape[0] * image.shape[1]

    MEAN64 = channel_sum / pixel_count
    VAR64 = channel_square_sum / pixel_count - np.square(MEAN64)

    MEAN = MEAN64.astype(np.float32)
    STD = np.sqrt(np.maximum(VAR64, 1e-12)).astype(np.float32)

    print("\nNormalization mean (development train only):", MEAN.tolist())
    print("Normalization std  (development train only):", STD.tolist())

    with open(
        OUT_DIR / "development_normalization.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump({
            "channel_order": list(INPUT_CHANNEL_ORDER),
            "mean": MEAN.tolist(),
            "std": STD.tolist(),
            "source": f"Group {source_group} development-train images only",
        }, f, indent=2)


    def get_raw(name):
        if name in SOURCE_CACHE:
            image, mask, stem = SOURCE_CACHE[name]

            return (
                image.astype(np.float32),
                mask.astype(np.float32),
                stem,
            )

        return load_raw(name)


    # =============================================================================
    # 9. DATASET / DATALOADER
    # =============================================================================

    class GDDataset(Dataset):
        def __init__(self, names, augment=False):
            self.names = list(names)
            self.augment = augment
            self.mean = MEAN.reshape(1, 1, 6)
            self.std = STD.reshape(1, 1, 6)

        def __len__(self):
            return len(self.names)

        def __getitem__(self, index):
            name = self.names[index]
            image, mask, stem = get_raw(name)

            if self.augment:
                if random.random() < 0.5:
                    image = np.flip(image, axis=1).copy()
                    mask = np.flip(mask, axis=1).copy()

                if random.random() < 0.5:
                    image = np.flip(image, axis=0).copy()
                    mask = np.flip(mask, axis=0).copy()

                if random.random() < 0.75:
                    gain = np.random.uniform(
                        0.88,
                        1.12,
                        size=(1, 1, 3),
                    ).astype(np.float32)

                    bias = np.random.uniform(
                        -0.035,
                        0.035,
                        size=(1, 1, 3),
                    ).astype(np.float32)

                    image[..., :3] = np.clip(
                        image[..., :3] * gain + bias,
                        0.0,
                        1.0,
                    )

            image = ((image - self.mean) / self.std).astype(np.float32)

            return (
                torch.from_numpy(image.transpose(2, 0, 1)).float(),
                torch.from_numpy(mask[None]).float(),
                stem,
            )


    def make_loader(
        names,
        augment=False,
        batch_size=1,
        shuffle=False,
        seed=None,
    ):
        generator = None

        if seed is not None:
            generator = torch.Generator()
            generator.manual_seed(seed)

        return DataLoader(
            GDDataset(names, augment=augment),
            batch_size=batch_size,
            shuffle=shuffle,
            generator=generator,
            num_workers=NUM_WORKERS,
            pin_memory=pin_memory,
            drop_last=False,
        )


    # =============================================================================
    # 10. EXACT SIX-CHANNEL PROPOSED MODEL
    # =============================================================================

    def group_norm(channels):
        groups = min(PREFERRED_GN_GROUPS, channels)

        while channels % groups:
            groups -= 1

        return nn.GroupNorm(
            groups,
            channels,
            eps=1e-5,
            affine=True,
        )


    class SCSE(nn.Module):
        def __init__(self, channels, reduction=16):
            super().__init__()

            hidden = max(channels // reduction, 1)

            self.channel = nn.Sequential(
                nn.AdaptiveAvgPool2d(1),
                nn.Conv2d(channels, hidden, 1),
                nn.SiLU(inplace=True),
                nn.Conv2d(hidden, channels, 1),
                nn.Sigmoid(),
            )

            self.spatial = nn.Sequential(
                nn.Conv2d(channels, 1, 1),
                nn.Sigmoid(),
            )

        def forward(self, x):
            return (
                x * self.channel(x)
                + x * self.spatial(x)
            )


    class ResBlock(nn.Module):
        def __init__(self, in_channels, out_channels, dropout=0.0):
            super().__init__()

            self.block = nn.Sequential(
                nn.Conv2d(
                    in_channels,
                    out_channels,
                    3,
                    padding=1,
                    bias=False,
                ),
                group_norm(out_channels),
                nn.SiLU(inplace=True),
                nn.Dropout2d(dropout) if dropout else nn.Identity(),
                nn.Conv2d(
                    out_channels,
                    out_channels,
                    3,
                    padding=1,
                    bias=False,
                ),
                group_norm(out_channels),
            )

            self.skip = (
                nn.Identity()
                if in_channels == out_channels
                else nn.Sequential(
                    nn.Conv2d(
                        in_channels,
                        out_channels,
                        1,
                        bias=False,
                    ),
                    group_norm(out_channels),
                )
            )

            self.attention = SCSE(out_channels)

        def forward(self, x):
            return self.attention(
                F.silu(
                    self.block(x) + self.skip(x),
                    inplace=True,
                )
            )


    class SoftGatedFusion(nn.Module):
        def __init__(self, channels):
            super().__init__()

            hidden = max(channels // 8, 8)

            self.gate = nn.Sequential(
                nn.Conv2d(
                    channels * 2,
                    hidden,
                    1,
                    bias=False,
                ),
                group_norm(hidden),
                nn.SiLU(inplace=True),
                nn.Conv2d(hidden, 2, 1),
            )

            self.refine = nn.Sequential(
                nn.Conv2d(
                    channels,
                    channels,
                    3,
                    padding=1,
                    bias=False,
                ),
                group_norm(channels),
                nn.SiLU(inplace=True),
            )

        def forward(self, rgb_feature, spectral_feature):
            weights = torch.softmax(
                self.gate(
                    torch.cat(
                        [rgb_feature, spectral_feature],
                        dim=1,
                    )
                ),
                dim=1,
            )

            fused = (
                weights[:, 0:1] * rgb_feature
                + weights[:, 1:2] * spectral_feature
            )

            return self.refine(fused), weights


    class UpBlock(nn.Module):
        def __init__(
            self,
            in_channels,
            skip_channels,
            out_channels,
            dropout=0.0,
        ):
            super().__init__()

            self.conv = ResBlock(
                in_channels + skip_channels,
                out_channels,
                dropout,
            )

        def forward(self, x, skip):
            x = F.interpolate(
                x,
                size=skip.shape[-2:],
                mode="bilinear",
                align_corners=False,
            )

            return self.conv(
                torch.cat([x, skip], dim=1)
            )


    class AuxiliarySupervisedGatedUNetNoNDRE(nn.Module):
        def __init__(self, base=28):
            super().__init__()

            c1, c2, c3, c4 = (
                base,
                base * 2,
                base * 4,
                base * 8,
            )

            self.pool = nn.MaxPool2d(2)

            self.rgb1 = ResBlock(3, c1)
            self.spec1 = ResBlock(3, c1)

            self.rgb2 = ResBlock(c1, c2)
            self.spec2 = ResBlock(c1, c2)

            self.rgb3 = ResBlock(c2, c3, 0.05)
            self.spec3 = ResBlock(c2, c3, 0.05)

            self.rgb4 = ResBlock(c3, c4, 0.10)
            self.spec4 = ResBlock(c3, c4, 0.10)

            self.fuse1 = SoftGatedFusion(c1)
            self.fuse2 = SoftGatedFusion(c2)
            self.fuse3 = SoftGatedFusion(c3)
            self.fuse4 = SoftGatedFusion(c4)

            self.rgb_auxiliary_head = nn.Conv2d(c4, 1, 1)
            self.spectral_auxiliary_head = nn.Conv2d(c4, 1, 1)

            self.center = ResBlock(c4, c4 * 2, 0.10)

            self.up4 = UpBlock(c4 * 2, c4, c4, 0.10)
            self.up3 = UpBlock(c4, c3, c3, 0.05)
            self.up2 = UpBlock(c3, c2, c2)
            self.up1 = UpBlock(c2, c1, c1)

            self.output = nn.Conv2d(c1, 1, 1)

        def modality_dropout(self, rgb, spectral):
            if (
                not self.training
                or MODALITY_DROPOUT_PROBABILITY <= 0
            ):
                return rgb, spectral

            draw = random.random()

            if draw < MODALITY_DROPOUT_PROBABILITY / 2:
                rgb = torch.zeros_like(rgb)
            elif draw < MODALITY_DROPOUT_PROBABILITY:
                spectral = torch.zeros_like(spectral)

            return rgb, spectral

        def forward(self, x, return_aux=False):
            if x.ndim != 4 or x.shape[1] != 6:
                raise ValueError(
                    f"Expected [B,6,H,W], received {tuple(x.shape)}"
                )

            input_size = x.shape[-2:]

            rgb, spectral = self.modality_dropout(
                x[:, :3],
                x[:, 3:],
            )

            r1 = self.rgb1(rgb)
            s1 = self.spec1(spectral)
            f1, _ = self.fuse1(r1, s1)

            r2 = self.rgb2(self.pool(r1 + f1))
            s2 = self.spec2(self.pool(s1 + f1))
            f2, _ = self.fuse2(r2, s2)

            r3 = self.rgb3(self.pool(r2 + f2))
            s3 = self.spec3(self.pool(s2 + f2))
            f3, _ = self.fuse3(r3, s3)

            r4 = self.rgb4(self.pool(r3 + f3))
            s4 = self.spec4(self.pool(s3 + f3))
            f4, _ = self.fuse4(r4, s4)

            center = self.center(self.pool(f4))

            d4 = self.up4(center, f4)
            d3 = self.up3(d4, f3)
            d2 = self.up2(d3, f2)
            d1 = self.up1(d2, f1)

            logits = self.output(d1)

            if return_aux:
                rgb_aux = F.interpolate(
                    self.rgb_auxiliary_head(r4),
                    size=input_size,
                    mode="bilinear",
                    align_corners=False,
                )

                spectral_aux = F.interpolate(
                    self.spectral_auxiliary_head(s4),
                    size=input_size,
                    mode="bilinear",
                    align_corners=False,
                )

                return logits, rgb_aux, spectral_aux

            return logits


    def build_model():
        model = AuxiliarySupervisedGatedUNetNoNDRE(
            BRANCH_BASE
        ).to(device)

        total = sum(
            p.numel()
            for p in model.parameters()
        )

        auxiliary = sum(
            p.numel()
            for name, p in model.named_parameters()
            if "auxiliary_head" in name
        )

        deployment = total - auxiliary

        print(
            f"Parameters: total={total:,}, "
            f"auxiliary={auxiliary:,}, "
            f"deployment={deployment:,}"
        )

        if total != 7_950_483:
            raise RuntimeError(
                f"Architecture mismatch: expected 7,950,483 parameters, got {total:,}."
            )

        return model


    # =============================================================================
    # 11. LOSS / METRICS
    # =============================================================================

    class ComboLoss(nn.Module):
        def forward(self, logits, targets):
            targets = targets.float()

            bce_map = F.binary_cross_entropy_with_logits(
                logits,
                targets,
                reduction="none",
            )

            bce = bce_map.mean()

            probabilities = torch.sigmoid(logits)

            pt = (
                probabilities * targets
                + (1.0 - probabilities) * (1.0 - targets)
            )

            focal = (
                bce_map * ((1.0 - pt) ** 2.0)
            ).mean()

            dims = tuple(
                range(1, logits.ndim)
            )

            intersection = (
                probabilities * targets
            ).sum(dims)

            denominator = (
                probabilities.sum(dims)
                + targets.sum(dims)
            )

            dice_loss = 1.0 - (
                (
                    2.0 * intersection
                    + 1e-6
                )
                /
                (
                    denominator
                    + 1e-6
                )
            ).mean()

            return (
                0.35 * bce
                + 0.25 * focal
                + 0.40 * dice_loss
            )


    criterion = ComboLoss()


    def auxiliary_loss_weights(epoch):
        progress = epoch / float(MAX_EPOCHS)

        if progress <= AUX_MIDDLE_PHASE_START:
            return (
                AUX_RGB_INITIAL_WEIGHT,
                AUX_SPECTRAL_INITIAL_WEIGHT,
            )

        if progress <= AUX_FINAL_PHASE_START:
            return (
                AUX_RGB_MIDDLE_WEIGHT,
                AUX_SPECTRAL_MIDDLE_WEIGHT,
            )

        return (
            AUX_RGB_FINAL_WEIGHT,
            AUX_SPECTRAL_FINAL_WEIGHT,
        )


    def confusion_numpy(probability, mask, threshold):
        pred = probability >= threshold
        target = mask >= 0.5

        return [
            int(np.logical_and(pred, target).sum()),
            int(np.logical_and(pred, ~target).sum()),
            int(np.logical_and(~pred, target).sum()),
            int(np.logical_and(~pred, ~target).sum()),
        ]


    def confusion_tensor(probability, mask, threshold):
        pred = probability >= threshold
        target = mask >= 0.5

        return [
            int((pred & target).sum().item()),
            int((pred & ~target).sum().item()),
            int((~pred & target).sum().item()),
            int((~pred & ~target).sum().item()),
        ]


    def add_confusion(a, b):
        return [
            int(x + y)
            for x, y in zip(a, b)
        ]


    def metric_dict(values):
        tp, fp, fn, tn = values
        e = 1e-7

        return {
            "dice": (2 * tp + e) / (2 * tp + fp + fn + e),
            "iou": (tp + e) / (tp + fp + fn + e),
            "precision": (tp + e) / (tp + fp + e),
            "recall": (tp + e) / (tp + fn + e),
            "accuracy": (tp + tn + e) / (tp + fp + fn + tn + e),
        }


    def save_csv(path, rows):
        if not rows:
            return

        with open(
            path,
            "w",
            newline="",
            encoding="utf-8",
        ) as f:
            writer = csv.DictWriter(
                f,
                fieldnames=list(rows[0].keys()),
            )

            writer.writeheader()
            writer.writerows(rows)


    # =============================================================================
    # 12. TRAINING HELPERS
    # =============================================================================

    def train_one_epoch(
        loader,
        model,
        optimizer,
        scaler,
        epoch,
    ):
        model.train()

        optimizer.zero_grad(
            set_to_none=True
        )

        rgb_aux_weight, spec_aux_weight = (
            auxiliary_loss_weights(epoch)
        )

        total_loss_sum = 0.0

        for step, (images, masks, _) in enumerate(
            tqdm(
                loader,
                desc=f"Train {epoch}",
                leave=False,
                disable=not SHOW_PROGRESS,
            ),
            start=1,
        ):
            images = images.to(
                device,
                non_blocking=True,
            )

            masks = masks.to(
                device,
                non_blocking=True,
            )

            with torch.amp.autocast(
                "cuda",
                enabled=device.type == "cuda",
            ):
                (
                    logits,
                    rgb_aux_logits,
                    spec_aux_logits,
                ) = model(
                    images,
                    return_aux=True,
                )

                main_loss = criterion(
                    logits,
                    masks,
                )

                rgb_aux_loss = criterion(
                    rgb_aux_logits,
                    masks,
                )

                spec_aux_loss = criterion(
                    spec_aux_logits,
                    masks,
                )

                total_loss = (
                    main_loss
                    + rgb_aux_weight * rgb_aux_loss
                    + spec_aux_weight * spec_aux_loss
                )

                scaled_loss = (
                    total_loss
                    / ACCUMULATION_STEPS
                )

            scaler.scale(
                scaled_loss
            ).backward()

            if (
                step % ACCUMULATION_STEPS == 0
                or step == len(loader)
            ):
                scaler.unscale_(optimizer)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    1.0,
                )

                scaler.step(optimizer)
                scaler.update()

                optimizer.zero_grad(
                    set_to_none=True
                )

            total_loss_sum += total_loss.item()

        return (
            total_loss_sum
            / max(len(loader), 1)
        )


    @torch.no_grad()
    def validate_coarse(loader, model):
        model.eval()

        totals = {
            threshold: [0, 0, 0, 0]
            for threshold in COARSE_THRESHOLDS
        }

        for images, masks, _ in tqdm(
            loader,
            desc="Development validation",
            leave=False,
            disable=not SHOW_PROGRESS,
        ):
            images = images.to(
                device,
                non_blocking=True,
            )

            masks = masks.to(
                device,
                non_blocking=True,
            )

            with torch.amp.autocast(
                "cuda",
                enabled=device.type == "cuda",
            ):
                probability = torch.sigmoid(
                    model(images)
                )

            for threshold in COARSE_THRESHOLDS:
                totals[threshold] = add_confusion(
                    totals[threshold],
                    confusion_tensor(
                        probability,
                        masks,
                        threshold,
                    ),
                )

        rows = []

        for threshold in COARSE_THRESHOLDS:
            row = metric_dict(
                totals[threshold]
            )

            row["threshold"] = float(threshold)

            rows.append(row)

        return max(
            rows,
            key=lambda row: row["dice"],
        )


    @torch.no_grad()
    def collect_predictions(
        loader,
        model,
        description,
    ):
        model.eval()

        records = []

        for images, masks, stems in tqdm(
            loader,
            desc=description,
            leave=False,
            disable=not SHOW_PROGRESS,
        ):
            images = images.to(
                device,
                non_blocking=True,
            )

            with torch.amp.autocast(
                "cuda",
                enabled=device.type == "cuda",
            ):
                probability = torch.sigmoid(
                    model(images)
                )

            records.append({
                "stem": stems[0],
                "probability": probability[
                    0, 0
                ].float().cpu().numpy(),
                "mask": masks[
                    0, 0
                ].float().numpy(),
            })

        return records


    def search_threshold(
        records,
        thresholds,
    ):
        rows = []

        for threshold in thresholds:
            total = [0, 0, 0, 0]

            for record in records:
                total = add_confusion(
                    total,
                    confusion_numpy(
                        record["probability"],
                        record["mask"],
                        threshold,
                    ),
                )

            row = metric_dict(total)
            row["threshold"] = float(threshold)

            rows.append(row)

        best = max(
            rows,
            key=lambda row: row["dice"],
        )

        return best, rows


    # =============================================================================
    # 13. DEVELOPMENT TRAINING ON SOURCE
    # =============================================================================

    seed_everything(SEED)

    dev_train_loader = make_loader(
        dev_train_names,
        augment=True,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
    )

    dev_val_loader = make_loader(
        dev_val_names,
        augment=False,
        batch_size=1,
        shuffle=False,
    )

    development_model = build_model()

    optimizer = torch.optim.AdamW(
        development_model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="max",
        factor=LR_REDUCTION_FACTOR,
        patience=LR_SCHEDULER_PATIENCE,
        min_lr=MIN_LEARNING_RATE,
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=device.type == "cuda",
    )

    best_development_dice = -1.0
    best_epoch = 0

    development_checkpoint = (
        OUT_DIR
        / "development_best_checkpoint.pt"
    )

    development_history = []
    development_lr_schedule = []

    for epoch in range(
        1,
        MAX_EPOCHS + 1,
    ):
        epoch_start = time.time()

        current_lr = float(
            optimizer.param_groups[0]["lr"]
        )

        development_lr_schedule.append(
            current_lr
        )

        train_loss = train_one_epoch(
            dev_train_loader,
            development_model,
            optimizer,
            scaler,
            epoch,
        )

        validation = validate_coarse(
            dev_val_loader,
            development_model,
        )

        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_dice": validation["dice"],
            "val_iou": validation["iou"],
            "val_precision": validation["precision"],
            "val_recall": validation["recall"],
            "coarse_threshold": validation["threshold"],
            "learning_rate": current_lr,
            "minutes": (time.time() - epoch_start) / 60.0,
        }

        development_history.append(row)
        save_csv(OUT_DIR / "development_training_history.csv", development_history)

        print(
            f"{direction} seed {SEED} DEV | epoch {epoch:03d} | "
            f"loss {train_loss:.4f} | "
            f"val Dice {validation['dice']:.4f} | "
            f"IoU {validation['iou']:.4f} | "
            f"th {validation['threshold']:.2f} | "
            f"lr {current_lr:.2e}"
        )

        if (
            validation["dice"]
            > best_development_dice
        ):
            best_development_dice = float(
                validation["dice"]
            )

            best_epoch = epoch

            torch.save({
                "model": development_model.state_dict(),
                "epoch": best_epoch,
                "best_development_dice": best_development_dice,
                "normalization_mean": MEAN.tolist(),
                "normalization_std": STD.tolist(),
                "seed": SEED,
                "direction": direction,
                "source_group": source_group,
                "source_count": len(source_names),
                "target_group": target_group,
                "target_count": len(target_names),
            }, development_checkpoint)

        scheduler.step(
            validation["dice"]
        )

        if (
            epoch - best_epoch
            >= EARLY_STOPPING_PATIENCE
        ):
            print(
                f"{direction}: early stopping after "
                f"{EARLY_STOPPING_PATIENCE} non-improving epochs."
            )
            break

    save_csv(
        OUT_DIR / "development_training_history.csv",
        development_history,
    )


    # =============================================================================
    # 14. EXACT VALIDATION THRESHOLD
    # =============================================================================

    try:
        checkpoint = torch.load(
            development_checkpoint,
            map_location=device,
            weights_only=False,
        )
    except TypeError:
        checkpoint = torch.load(
            development_checkpoint,
            map_location=device,
        )

    development_model.load_state_dict(
        checkpoint["model"]
    )

    validation_records = collect_predictions(
        dev_val_loader,
        development_model,
        f"{direction}: exact threshold validation",
    )

    fine_validation_best, fine_threshold_rows = (
        search_threshold(
            validation_records,
            FINE_THRESHOLDS,
        )
    )

    selected_threshold = float(
        fine_validation_best["threshold"]
    )

    save_csv(
        OUT_DIR / "development_threshold_search.csv",
        fine_threshold_rows,
    )

    print("\n" + "-" * 80)
    print(f"{direction} seed {SEED} DEVELOPMENT SELECTION")
    print("-" * 80)
    print(
        "Best epoch              :",
        best_epoch,
    )
    print(
        "Best development Dice   :",
        f"{best_development_dice:.4f}",
    )
    print(
        "Exact selected threshold:",
        f"{selected_threshold:.2f}",
    )
    print(
        "Threshold-val Dice      :",
        f"{fine_validation_best['dice']:.4f}",
    )


    # =============================================================================
    # 15. DISCARD DEV MODEL AND FRESH REFIT ON ALL SOURCE IMAGES
    # =============================================================================

    del development_model
    del optimizer
    del scaler
    del scheduler
    del checkpoint
    del validation_records

    if device.type == "cuda":
        torch.cuda.empty_cache()

    seed_everything(SEED)

    final_train_loader = make_loader(
        source_names,
        augment=True,
        batch_size=BATCH_SIZE,
        shuffle=True,
        seed=SEED,
    )

    final_model = build_model()

    final_optimizer = torch.optim.AdamW(
        final_model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    final_scaler = torch.amp.GradScaler(
        "cuda",
        enabled=device.type == "cuda",
    )

    final_history = []

    for epoch in range(
        1,
        best_epoch + 1,
    ):
        # Replay the LR that the development run used at this epoch.
        replay_lr = float(
            development_lr_schedule[
                min(
                    epoch - 1,
                    len(development_lr_schedule) - 1,
                )
            ]
        )

        for group in final_optimizer.param_groups:
            group["lr"] = replay_lr

        train_loss = train_one_epoch(
            final_train_loader,
            final_model,
            final_optimizer,
            final_scaler,
            epoch,
        )

        final_history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "learning_rate": replay_lr,
        })

        save_csv(OUT_DIR / "final_refit_history.csv", final_history)

        print(
            f"{direction} seed {SEED} FINAL REFIT | "
            f"epoch {epoch:03d}/{best_epoch:03d} | "
            f"loss {train_loss:.4f} | "
            f"lr {replay_lr:.2e}"
        )

    save_csv(
        OUT_DIR / "final_refit_history.csv",
        final_history,
    )

    final_checkpoint = (
        OUT_DIR
        / "final_all_source_checkpoint.pt"
    )

    torch.save({
        "model": final_model.state_dict(),
        "train_all_source_count": len(source_names),
        "seed": SEED,
        "direction": direction,
        "source_group": source_group,
        "target_group": target_group,
        "heldout_target_count": len(target_names),
        "selected_epoch_from_development": best_epoch,
        "selected_threshold_from_development": selected_threshold,
        "normalization_mean": MEAN.tolist(),
        "normalization_std": STD.tolist(),
        "channel_order": list(INPUT_CHANNEL_ORDER),
    }, final_checkpoint)


    # =============================================================================
    # 16. UNTOUCHED TARGET-GROUP TEST
    # =============================================================================

    test_loader = make_loader(
        target_names,
        augment=False,
        batch_size=1,
        shuffle=False,
    )

    test_records = collect_predictions(
        test_loader,
        final_model,
        f"{direction}: FINAL HELD-OUT TEST",
    )

    global_values = [0, 0, 0, 0]
    per_image_rows = []

    for record in test_records:
        values = confusion_numpy(
            record["probability"],
            record["mask"],
            selected_threshold,
        )

        global_values = add_confusion(
            global_values,
            values,
        )

        row = metric_dict(values)

        row.update({
            "stem": record["stem"],
            "threshold": selected_threshold,
            "tp": values[0],
            "fp": values[1],
            "fn": values[2],
            "tn": values[3],
        })

        per_image_rows.append(row)

    final_metrics = metric_dict(
        global_values
    )

    summary = {
        "direction": direction,
        "seed": SEED,
        "source_group": source_group,
        "target_group": target_group,
        "source_count": len(source_names),
        "group_A_count": len(GROUP_A),
        "group_B_count": len(GROUP_B),
        "development_train_count": len(dev_train_names),
        "development_validation_count": len(dev_val_names),
        "target_test_count": len(target_names),
        "minimum_A_B_center_distance_m": minimum_ab_distance_m,
        "best_epoch": best_epoch,
        "best_development_dice": best_development_dice,
        "selected_threshold": selected_threshold,
        "threshold_validation_dice": fine_validation_best["dice"],
        "test_dice": final_metrics["dice"],
        "test_iou": final_metrics["iou"],
        "test_precision": final_metrics["precision"],
        "test_recall": final_metrics["recall"],
        "test_accuracy": final_metrics["accuracy"],
        "tp": global_values[0],
        "fp": global_values[1],
        "fn": global_values[2],
        "tn": global_values[3],
    }

    save_csv(
        OUT_DIR / "final_test_metrics.csv",
        [summary],
    )

    save_csv(
        OUT_DIR / "per_image_test_metrics.csv",
        per_image_rows,
    )

    with open(
        OUT_DIR / "experiment_configuration.json",
        "w",
        encoding="utf-8",
    ) as f:
        json.dump({
            "dataset": str(BASE_DIR),
            "protocol": f"cross-acquisition {direction}",
            "source_notebook_cell": "gd-second.ipynb, recheck cell (zero-based 12)",
            "metric_aggregation": "pooled TP/FP/FN/TN over held-out images",
            "development_split": "sklearn train_test_split, 20 percent rounded, random_state=seed",
            "normalization_source": "source development-training subset only; reused for refit and test",
            "primary_inference": "no TTA",
            "threshold_grid": FINE_THRESHOLDS,
            "refit_lr_schedule": development_lr_schedule[:best_epoch],
            "run_signature": run_signature,
            "grouping": "KMeans(n_clusters=2) on GPS camera centres; larger cluster named A",
            "seed": SEED,
            "target_size": list(TARGET_SIZE),
            "batch_size": BATCH_SIZE,
            "accumulation_steps": ACCUMULATION_STEPS,
            "max_development_epochs": MAX_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "optimizer": "AdamW",
            "learning_rate": LEARNING_RATE,
            "weight_decay": WEIGHT_DECAY,
            "lr_scheduler": "ReduceLROnPlateau",
            "lr_scheduler_patience": LR_SCHEDULER_PATIENCE,
            "lr_reduction_factor": LR_REDUCTION_FACTOR,
            "minimum_lr": MIN_LEARNING_RATE,
            "channels": list(INPUT_CHANNEL_ORDER),
            "best_epoch": best_epoch,
            "selected_threshold": selected_threshold,
            "normalization_mean": MEAN.tolist(),
            "normalization_std": STD.tolist(),
            "final_test_metrics": summary,
        }, f, indent=2)


    # =============================================================================
    # 17. FINAL OUTPUT
    # =============================================================================

    print("\n" + "=" * 80)
    print(f"{direction} seed {SEED} FINAL HELD-OUT TEST RESULT")
    print("=" * 80)
    print("Train all source :", len(source_names))
    print("Test all target  :", len(target_names))
    print("Best epoch       :", best_epoch)
    print("Best dev Dice    :", f"{best_development_dice:.4f}")
    print("Threshold        :", f"{selected_threshold:.2f}")
    print("Dice             :", f"{final_metrics['dice']:.4f}")
    print("IoU              :", f"{final_metrics['iou']:.4f}")
    print("Precision        :", f"{final_metrics['precision']:.4f}")
    print("Recall           :", f"{final_metrics['recall']:.4f}")
    print("Accuracy         :", f"{final_metrics['accuracy']:.4f}")
    print("Checkpoint       :", final_checkpoint)


    # Completion marker is written only after final metrics and configuration exist.
    with open(OUT_DIR / "completed_run.json", "w", encoding="utf-8") as f:
        json.dump({"run_signature": run_signature, "metrics": summary}, f, indent=2)
    return summary


class Tee:
    """Keep printed output in the notebook and the per-run log file."""
    def __init__(self, *streams):
        self.streams = streams

    def write(self, data):
        for stream in self.streams:
            stream.write(data)
            stream.flush()
        return len(data)

    def flush(self):
        for stream in self.streams:
            stream.flush()


def write_summary(rows, path):
    if not rows:
        return
    keys = list(dict.fromkeys(key for row in rows for key in row))
    temporary = path.with_suffix(".csv.tmp")
    with open(temporary, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=keys)
        writer.writeheader()
        writer.writerows(rows)
    temporary.replace(path)


def load_existing_results():
    """Import only explicitly supplied full-precision A->B CSV rows."""
    rows = []
    for path in EXISTING_RESULT_CSVS:
        with open(path, newline="", encoding="utf-8-sig") as f:
            for raw in csv.DictReader(f):
                direction = raw.get("direction", raw.get("run", ""))
                seed = int(raw["seed"])
                if direction != "A_to_B" or seed not in (42, 123):
                    raise ValueError(f"Existing results must be A_to_B seed 42/123: {path}")
                row = {"direction": direction, "seed": seed}
                for metric in ("dice", "iou", "precision", "recall", "accuracy"):
                    value = raw.get("test_" + metric, raw.get(metric))
                    if value is None:
                        raise ValueError(f"Missing {metric} in {path}; do not estimate it.")
                    row["test_" + metric] = float(value)
                source_count = raw.get("source_count", raw.get("group_A_source_count", raw.get("final_train_images")))
                target_count = raw.get("target_test_count", raw.get("group_B_test_count", raw.get("heldout_test_images")))
                if int(source_count or -1) != 217 or int(target_count or -1) != 183:
                    raise ValueError(f"Existing A_to_B counts must be train=217, test=183: {path}")
                row.update({"source_count": 217, "target_test_count": 183,
                            "provenance": str(path), "protocol_source": "external CSV; compare saved configs before pooling"})
                rows.append(row)
    return rows


def save_and_print_summary(rows):
    write_summary(rows, MASTER_OUT / "cross_acquisition_summary.csv")
    print("\nCROSS-ACQUISITION SUMMARY (actual completed/imported rows only)")
    for row in sorted(rows, key=lambda r: (r["direction"], int(r["seed"]))):
        print(f"{row['direction']} | seed {int(row['seed'])} | "
              f"train {row['source_count']} | test {row['target_test_count']} | "
              f"Dice {float(row['test_dice']):.6f} | IoU {float(row['test_iou']):.6f} | "
              f"Precision {float(row['test_precision']):.6f} | Recall {float(row['test_recall']):.6f}")
    print("Summary CSV:", MASTER_OUT / "cross_acquisition_summary.csv")


def code_fingerprint(code):
    """Ignore notebook cell filenames when recognizing an unchanged runner."""
    import types
    constants = [code_fingerprint(v) if isinstance(v, types.CodeType) else repr(v)
                 for v in code.co_consts]
    payload = [code.co_code.hex(), code.co_names, code.co_varnames,
               code.co_freevars, code.co_cellvars, code.co_flags,
               code.co_argcount, code.co_kwonlyargcount, constants]
    return hashlib.sha256(json.dumps(payload).encode()).hexdigest()


def main():
    if REQUIRE_CUDA and not torch.cuda.is_available():
        raise RuntimeError("Enable a GPU accelerator before running this training script.")
    if len(RUN_PLAN) != len(set(RUN_PLAN)):
        raise ValueError("RUN_PLAN contains duplicate direction/seed pairs.")
    if any(d not in ("A_to_B", "B_to_A") or s not in (42, 123, 2026) for d, s in RUN_PLAN):
        raise ValueError("Use A_to_B/B_to_A with seed 42, 123 or 2026.")
    MASTER_OUT.mkdir(parents=True, exist_ok=True)
    rows = load_existing_results()
    if len({(r['direction'], r['seed']) for r in rows}) != len(rows):
        raise ValueError("Duplicate existing result rows supplied.")
    if {(r['direction'], r['seed']) for r in rows} & set(RUN_PLAN):
        raise ValueError("Existing rows conflict with RUN_PLAN.")
    print("Group A = 217 images; Group B = 183 images.")
    print("New experiments:", RUN_PLAN)
    print("Each experiment = development training + fresh all-source refit.")
    # Guards completed-run reuse after changes to code, hyperparameters or paths.
    core_fingerprint = code_fingerprint(run_experiment.__code__)
    config = {
        "core": core_fingerprint, "dataset_root": str(DATASET_ROOT),
        "size": TARGET_SIZE, "batch": BATCH_SIZE, "accum": ACCUMULATION_STEPS,
        "epochs": MAX_EPOCHS, "patience": EARLY_STOPPING_PATIENCE,
        "lr": LEARNING_RATE, "wd": WEIGHT_DECAY,
        "lr_factor": LR_REDUCTION_FACTOR, "lr_patience": LR_SCHEDULER_PATIENCE,
        "min_lr": MIN_LEARNING_RATE, "workers": NUM_WORKERS,
        "cache": CACHE_SOURCE_GROUP, "base": BRANCH_BASE,
        "gn": PREFERRED_GN_GROUPS, "modality_dropout": MODALITY_DROPOUT_PROBABILITY,
        "channels": INPUT_CHANNEL_ORDER, "mask_color": MASK_POSITIVE_RGB.tolist(),
        "aux": [AUX_RGB_INITIAL_WEIGHT, AUX_SPECTRAL_INITIAL_WEIGHT,
                AUX_RGB_MIDDLE_WEIGHT, AUX_SPECTRAL_MIDDLE_WEIGHT,
                AUX_RGB_FINAL_WEIGHT, AUX_SPECTRAL_FINAL_WEIGHT,
                AUX_MIDDLE_PHASE_START, AUX_FINAL_PHASE_START],
        "coarse": COARSE_THRESHOLDS, "fine": FINE_THRESHOLDS,
    }
    for direction, seed in RUN_PLAN:
        out_dir = MASTER_OUT / f"{direction}_seed{seed}"
        out_dir.mkdir(parents=True, exist_ok=True)
        signature = hashlib.sha256(json.dumps({**config, "direction": direction, "seed": seed}, sort_keys=True).encode()).hexdigest()
        marker = out_dir / "completed_run.json"
        try:
            if SKIP_COMPLETED and marker.exists():
                saved = json.loads(marker.read_text())
                if saved["run_signature"] != signature:
                    raise RuntimeError(f"Code/settings changed for {out_dir.name}; use a new MASTER_OUT.")
                row = saved["metrics"]
                print("Reusing completed result:", out_dir.name)
            else:
                with open(out_dir / "console_log.txt", "a", encoding="utf-8") as log:
                    with contextlib.redirect_stdout(Tee(sys.stdout, log)):
                        row = run_experiment(direction, seed, out_dir, signature)
            rows.append(row)
            save_and_print_summary(rows)
            archive = shutil.make_archive(str(MASTER_OUT / out_dir.name) + "_results", "zip", str(out_dir))
            print("Completed-run ZIP:", archive)
        except Exception:
            (out_dir / "last_error.txt").write_text(traceback.format_exc())
            save_and_print_summary(rows)
            raise
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
    print("\nAll requested new experiments finished.")
    print("Existing A->B seeds 42/123 are omitted unless supplied in EXISTING_RESULT_CSVS.")
    print("Download the metrics and ZIPs before ending the session.")


if __name__ == "__main__":
    main()
